In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras import layers

from utils import load_images

I0000 00:00:1787930886.440302   15637 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787930888.616521   15637 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787930893.789047   15637 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
x_train = load_images('data/train-images.idx3-ubyte')
x_train.shape

(60000, 28, 28)

In [3]:
def build_generator():
    
    model = Sequential([
        layers.Input(shape=(100,)),
        
        layers.Dense(7 * 7 * 128),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        
        layers.Reshape((7, 7, 128)),
        
        layers.Conv2DTranspose(
            64, kernel_size=4, strides=2, padding='same'
        ),
        
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        
        layers.Conv2DTranspose(
            1, kernel_size=4, strides=2, padding='same', activation='tanh'
        )
    ])
    
    return model

In [4]:
def build_discriminator():
    model = Sequential([
        layers.Input(shape=(28, 28, 1,)),
        
        layers.Conv2D(
            64, kernel_size=4, strides=2, padding='same',
        ),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        
        layers.Conv2D(
            128, kernel_size=4, strides=2, padding='same',
        ),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        
        layers.Flatten(),
        
        layers.Dense(1)
        
    ])
    
    return model

In [5]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(real_output, fake_output):
    
    real_loss = cross_entropy(
        tf.ones_like(real_output), # Expected
        real_output # Actual
    )
    
    fake_loss = cross_entropy(
    tf.zeros_like(fake_output),
    fake_output
    )
    
    return real_loss + fake_loss

def generator_loss(fake_output):

    return cross_entropy(
        tf.ones_like(fake_output),
        fake_output
    )

In [6]:
@tf.function
def train_step(
    images,
    generator,
    discriminator,
    generator_loss,
    discriminator_loss,
    generator_optimizer,
    discriminator_optimizer
):
    
    noise = tf.random.normal([images.shape[0], 100])
    
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        
        generated_images = generator(noise, training=True)
        
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)
        
    gen_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    
    generator_optimizer.apply_gradients(
        zip(gen_gradients, generator.trainable_variables)
    )
    
    discriminator_optimizer.apply_gradients(
        zip(disc_gradients, discriminator.trainable_variables)
    )
        
    return gen_loss, disc_loss

In [7]:
x_train=x_train[..., np.newaxis]

x_train = x_train.astype("float32")

x_train = (x_train - 127.5) / 127.5

In [8]:
x_train.shape

(60000, 28, 28, 1)

In [10]:

generator = build_generator()
discriminator = build_discriminator()

generator_optimizer = tf.keras.optimizers.Adam(0.0002)
discriminator_optimizer = tf.keras.optimizers.Adam(0.0002)

BATCH_SIZE = 64

dataset = tf.data.Dataset.from_tensor_slices(x_train)

dataset = dataset.shuffle(
    len(x_train)
).batch(
    BATCH_SIZE
)

E0000 00:00:1787930966.735146   15637 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1787930966.741282   15988 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1787930966.771531   15637 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
W0000 00:00:1787930967.922467   15637 cpu_allocator_impl.cc:82] Allocation of 188160000 exceeds 10% of free system memory.


In [13]:
images = next(iter(dataset))

for images in dataset:
    gen_loss, disc_loss = train_step(
        images,
        generator,
        discriminator,
        generator_loss,
        discriminator_loss,
        generator_optimizer,
        discriminator_optimizer
    )

print("Generator loss:", gen_loss.numpy())
print("Discriminator loss:", disc_loss.numpy())

KeyboardInterrupt: 